# Analysis of supplementary experiment: observers P1 and P2

## Subsampling analysis

In [1]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sklearn
import ast

import cblearn

from load_data import load_all, DATA_ROOT
from utils import calculate_triplet_correct
from dimensionality_estimation import estimate_dimensionality
from cblearn.preprocessing import triplets_from_oddoneout
from cblearn.utils import check_query_response

print('cblearn:', cblearn.__version__)
print('sklearn:', sklearn.__version__)

nan = np.nan

# seeds the SOE initialisations and the cross-validation splits of every fit
SEED = 0

cblearn: 0.3.0
sklearn: 1.5.2


# Reading data

### reading number of images in each set

In [2]:
set_names = ('action', 'greebles', 'taxonomic')
objects = {}
for name in set_names:
    df = pd.read_csv(DATA_ROOT / f'resources/{name}_info.csv')
    df['image_set'] = name
    objects[name] = df
    print(f"{name}: {len(df)} images")

action: 18 images
greebles: 18 images
taxonomic: 18 images


### reading embedding data done by notebook embedding_analysis_P1_P2.ipynb

In [3]:
joined_df = pd.read_csv('../data/found-dimensionality-embeddings-aggregated-per-observer-nsplit-5-nrepeat-30.csv')

In [4]:
joined_df = joined_df[(joined_df['observer']=='P1') | (joined_df['observer']=='P2')]

In [5]:
joined_df = joined_df[(joined_df['task']=='classic') | (joined_df['task']=='odd one out')]

In [6]:
joined_df

,observer,task,image_set,embedding_ready_triplets,raw_triplets,responses,Nt,dimensionality,embedding,triplets
45,P1,classic,action,[[ 3 16 14]\n [ 6 5 10]\n [ 1 8 15]\n ...\n ...,"[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, ...",2445,2,"[[-0.06934600655537254, 0.23346518275308703], ...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1..."
46,P1,classic,greebles,[[ 0 7 12]\n [ 5 10 8]\n [10 9 16]\n ...\n ...,"[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, 1, -1, 1...",2435,2,"[[0.07602326347126374, 0.23507685838587017], [...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1..."
47,P1,classic,taxonomic,[[ 0 1 3]\n [ 0 1 5]\n [ 0 12 9]\n ...\n ...,"[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[1, 1, 1, 1, 1, -1, 1, 1, -1, -1, -1, -1, -1, ...",2436,2,"[[-0.09584964415716074, -0.45110414211323197],...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ..."
48,P1,odd one out,action,[[15. 17. 9.]\n [ 3. 6. 5.]\n [17. 8. 11.]...,"[[9.0, 15.0, 17.0], [3.0, 5.0, 6.0], [17.0, 8....","[0, 1, 2, 2, 0, 2, 2, 1, 1, 0, 2, 1, 2, 1, 2, ...",4894,2,"[[0.21791835293914694, 0.166012000508984], [0....","[[15.0, 17.0, 9.0], [3.0, 6.0, 5.0], [17.0, 8...."
49,P1,odd one out,greebles,[[ 5. 10. 1.]\n [ 4. 9. 7.]\n [ 7. 6. 13.]...,"[[5.0, 1.0, 10.0], [4.0, 7.0, 9.0], [7.0, 6.0,...","[1, 1, 2, 0, 2, 0, 1, 0, 0, 1, 1, 0, 2, 0, 1, ...",4884,1,"[[-0.30218815704419694], [-0.30218818756681903...","[[5.0, 10.0, 1.0], [4.0, 9.0, 7.0], [7.0, 6.0,..."
50,P1,odd one out,taxonomic,[[ 1. 16. 3.]\n [10. 3. 0.]\n [ 6. 10. 15.]...,"[[1.0, 3.0, 16.0], [10.0, 0.0, 3.0], [15.0, 6....","[1, 1, 0, 1, 1, 2, 2, 0, 2, 0, 0, 1, 2, 1, 1, ...",4894,1,"[[-0.30080522948785315], [-0.30050482645990156...","[[1.0, 16.0, 3.0], [10.0, 3.0, 0.0], [6.0, 10...."
54,P2,classic,action,[[ 3 16 14]\n [ 6 5 10]\n [ 1 8 15]\n ...\n ...,"[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -...",2442,2,"[[-0.5246048711985275, 0.1532674894858705], [-...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1..."
55,P2,classic,greebles,[[ 0 7 12]\n [ 5 10 8]\n [10 9 16]\n ...\n ...,"[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[1, -1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1...",2425,2,"[[-0.5390362898896611, -0.06426774720972277], ...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1..."
56,P2,classic,taxonomic,[[ 0 1 3]\n [ 0 1 5]\n [ 0 12 9]\n ...\n ...,"[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[1, 1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, -1, -...",2439,3,"[[0.4463779236554152, 0.2666207605693572, -0.3...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ..."
57,P2,odd one out,action,[[ 3. 4. 14.]\n [17. 14. 9.]\n [ 3. 4. 2.]...,"[[14.0, 3.0, 4.0], [17.0, 9.0, 14.0], [2.0, 3....","[0, 1, 0, 2, 1, 2, 0, 0, 0, 2, 2, 0, 1, 0, 2, ...",4838,1,"[[0.6641610145836546], [0.6758773819778607], [...","[[3.0, 4.0, 14.0], [17.0, 14.0, 9.0], [3.0, 4...."


In [7]:
joined_df['raw_triplets'] = joined_df['raw_triplets'].apply(lambda x: ast.literal_eval(x))
joined_df['responses'] = joined_df['responses'].apply(lambda x: ast.literal_eval(x))
joined_df['embedding'] = joined_df['embedding'].apply(lambda x: ast.literal_eval(x))

In [8]:
joined_df

,observer,task,image_set,embedding_ready_triplets,raw_triplets,responses,Nt,dimensionality,embedding,triplets
45,P1,classic,action,[[ 3 16 14]\n [ 6 5 10]\n [ 1 8 15]\n ...\n ...,"[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, -1, -1, ...",2445,2,"[[-0.06934600655537254, 0.23346518275308703], ...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1..."
46,P1,classic,greebles,[[ 0 7 12]\n [ 5 10 8]\n [10 9 16]\n ...\n ...,"[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[1, -1, 1, 1, -1, 1, 1, 1, 1, -1, -1, 1, -1, 1...",2435,2,"[[0.07602326347126374, 0.23507685838587017], [...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1..."
47,P1,classic,taxonomic,[[ 0 1 3]\n [ 0 1 5]\n [ 0 12 9]\n ...\n ...,"[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[1, 1, 1, 1, 1, -1, 1, 1, -1, -1, -1, -1, -1, ...",2436,2,"[[-0.09584964415716074, -0.45110414211323197],...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ..."
48,P1,odd one out,action,[[15. 17. 9.]\n [ 3. 6. 5.]\n [17. 8. 11.]...,"[[9.0, 15.0, 17.0], [3.0, 5.0, 6.0], [17.0, 8....","[0, 1, 2, 2, 0, 2, 2, 1, 1, 0, 2, 1, 2, 1, 2, ...",4894,2,"[[0.21791835293914694, 0.166012000508984], [0....","[[15.0, 17.0, 9.0], [3.0, 6.0, 5.0], [17.0, 8...."
49,P1,odd one out,greebles,[[ 5. 10. 1.]\n [ 4. 9. 7.]\n [ 7. 6. 13.]...,"[[5.0, 1.0, 10.0], [4.0, 7.0, 9.0], [7.0, 6.0,...","[1, 1, 2, 0, 2, 0, 1, 0, 0, 1, 1, 0, 2, 0, 1, ...",4884,1,"[[-0.30218815704419694], [-0.30218818756681903...","[[5.0, 10.0, 1.0], [4.0, 9.0, 7.0], [7.0, 6.0,..."
50,P1,odd one out,taxonomic,[[ 1. 16. 3.]\n [10. 3. 0.]\n [ 6. 10. 15.]...,"[[1.0, 3.0, 16.0], [10.0, 0.0, 3.0], [15.0, 6....","[1, 1, 0, 1, 1, 2, 2, 0, 2, 0, 0, 1, 2, 1, 1, ...",4894,1,"[[-0.30080522948785315], [-0.30050482645990156...","[[1.0, 16.0, 3.0], [10.0, 3.0, 0.0], [6.0, 10...."
54,P2,classic,action,[[ 3 16 14]\n [ 6 5 10]\n [ 1 8 15]\n ...\n ...,"[[3.0, 14.0, 16.0], [6.0, 5.0, 10.0], [1.0, 8....","[-1, 1, 1, -1, -1, -1, 1, 1, 1, 1, 1, 1, -1, -...",2442,2,"[[-0.5246048711985275, 0.1532674894858705], [-...","[[3, 16, 14], [6, 5, 10], [1, 8, 15], [2, 4, 1..."
55,P2,classic,greebles,[[ 0 7 12]\n [ 5 10 8]\n [10 9 16]\n ...\n ...,"[[0.0, 7.0, 12.0], [5.0, 8.0, 10.0], [10.0, 9....","[1, -1, 1, 1, -1, 1, 1, -1, 1, -1, 1, 1, -1, 1...",2425,2,"[[-0.5390362898896611, -0.06426774720972277], ...","[[0, 7, 12], [5, 10, 8], [10, 9, 16], [0, 7, 1..."
56,P2,classic,taxonomic,[[ 0 1 3]\n [ 0 1 5]\n [ 0 12 9]\n ...\n ...,"[[0.0, 1.0, 3.0], [0.0, 1.0, 5.0], [0.0, 12.0,...","[1, 1, 1, 1, 1, -1, 1, 1, 1, -1, -1, -1, -1, -...",2439,3,"[[0.4463779236554152, 0.2666207605693572, -0.3...","[[0, 1, 3], [0, 1, 5], [0, 12, 9], [6, 3, 9], ..."
57,P2,odd one out,action,[[ 3. 4. 14.]\n [17. 14. 9.]\n [ 3. 4. 2.]...,"[[14.0, 3.0, 4.0], [17.0, 9.0, 14.0], [2.0, 3....","[0, 1, 0, 2, 1, 2, 0, 0, 0, 2, 2, 0, 1, 0, 2, ...",4838,1,"[[0.6641610145836546], [0.6758773819778607], [...","[[3.0, 4.0, 14.0], [17.0, 14.0, 9.0], [3.0, 4...."


# Calculating triplet accuracy from subsampling of triplets 


In [9]:
# Logic:
# we take a subsample of the data of size n
# fit an embedding with them, keeping the target dimensionality we have found in the whole dataset
# see how many of the complete set of triplets this embedding correctly predicts -> triplet accuracy
# repeat for increasing n

# we should expect that as n increases, the triplet accuracy should steadily increase reaching a maximum that is the same 
# as the one with the full set of triplets.
# how quickly ascends is the interesting question. Does it do it rapidely and then stays in a plateau? that would indicate that 
# there is a critical point at which more triplets just give diminishing returns.
# Alternatively it could be that the increase is always steady, maybe close to linear. This would indicate that there is no critical 
# point and that more triplets always give you more information of equal importance

In [10]:
def raw_triplets_to_embedding_ready_triplets(triplets, responses, task):
    if task == 'classic':
        triplets = check_query_response(triplets, responses, 'list-order')
    elif task == 'odd one out':
        triplets = triplets_from_oddoneout(triplets, responses.astype(int))

    return triplets

In [11]:
do_simulation = False

# for each embedding: observer, imageset and task
n_runs = 100

data_all = []
for ix, d in joined_df.iterrows():

    obs = d['observer']
    image_set = d['image_set']
    task = d['task']
    
    # we take raw triplets, as to simulate an actual experiment 
    raw_triplets = np.array(d['raw_triplets']).astype('int')
    responses = np.array(d['responses']).astype('int')
    Nt = len(raw_triplets)
    embedding = np.array(d['embedding'])
    dimensionality = d['dimensionality']

    
    triplets_all = raw_triplets_to_embedding_ready_triplets(raw_triplets, responses, task)
    acc_all = calculate_triplet_correct(embedding, triplets_all)
        
    print(f"{obs} - {task} - {image_set} - {len(raw_triplets)} triplets - dimensionality full dataset: {dimensionality} - accuracy: {acc_all}")
    data_all.append([obs, task, image_set, dimensionality, acc_all, Nt])

    #######
    if do_simulation:
        n_subsamples = np.array([28, 56, 112, 225, 450, 900, 1800])
    
        data = []
        # iterate across different subsample lengths
        for n in n_subsamples: 
    
             # sets a fixed random seed to make this analysis reproducible
            np.random.seed(0)
            # across runs we still get different random samples of triplets to do the training,
            # we just fix this at the begining of the loop 
            
            # loop over different number of random shuffles 
            for nr in range(n_runs):
                
                goodsample=False
                while not goodsample:
                    
                    idx = np.random.choice(np.arange(0, Nt), size=n, replace=False)
                    #  take the triplet sample of size n           
                    raw_triplets_train = raw_triplets[idx]
                    responses_train = responses[idx]
    
                    # we need to check that the triplets we have contains at least one every stimuli, otherwise we need to sample again
                    # this is particularly relevant for low sample sizes
                    nstim_sample = len(np.unique(raw_triplets_train.ravel()))
                    
                    if nstim_sample == 18:
                        goodsample = True
    
                # we compute the embedding ready triplets depending on the task
                triplets_train = raw_triplets_to_embedding_ready_triplets(raw_triplets_train, responses_train, task)
                
                # estimate dimensionality and resulting embedding from the training data 
                # dimensions 1 to 5, 5 splits x 30 repeats; SOE and the cross-validation splits seeded with SEED
                dim_result, embedding_sample = estimate_dimensionality(triplets_train, random_state=SEED)
                dim_sample = dim_result.estimated_dimension
                
                # now we use the rest to calcualte triplet error 
                idx_test = np.setdiff1d(np.arange(0, Nt), idx)
    
                raw_triplets_test = raw_triplets[idx_test]
                responses_test = responses[idx_test]
                
                triplets_test = raw_triplets_to_embedding_ready_triplets(raw_triplets_test, responses_test, task)
                    
                # calculate triplet accuracy
                acc = calculate_triplet_correct(embedding_sample, triplets_test)
    
                print(f"run #{nr} - ntriplets in sample: {n} - d: {dim_sample} - accuracy: {acc}")
    
                data.append([nr, n, dim_sample, acc])
    
        df_triplet_accuracy = pd.DataFrame(data, columns = ['random-run', 'n', 'd', 'accuracy'])
        df_triplet_accuracy['observer'] = obs
        df_triplet_accuracy['task'] = task
        df_triplet_accuracy['image_set'] = image_set
        df_triplet_accuracy.to_csv(f"../data/subsampling_analysis-{obs}-{task}-{image_set}-{n_runs}.csv", index=False)

P1 - classic - action - 2445 triplets - dimensionality full dataset: 2 - accuracy: 0.9198364008179959
P1 - classic - greebles - 2435 triplets - dimensionality full dataset: 2 - accuracy: 0.7827515400410677
P1 - classic - taxonomic - 2436 triplets - dimensionality full dataset: 2 - accuracy: 0.9359605911330049
P1 - odd one out - action - 2447 triplets - dimensionality full dataset: 2 - accuracy: 0.9871270944013077
P1 - odd one out - greebles - 2442 triplets - dimensionality full dataset: 1 - accuracy: 0.8452088452088452
P1 - odd one out - taxonomic - 2447 triplets - dimensionality full dataset: 1 - accuracy: 0.9969350224765018
P2 - classic - action - 2442 triplets - dimensionality full dataset: 2 - accuracy: 0.8226863226863227
P2 - classic - greebles - 2425 triplets - dimensionality full dataset: 2 - accuracy: 0.8412371134020619
P2 - classic - taxonomic - 2439 triplets - dimensionality full dataset: 3 - accuracy: 0.8831488314883149
P2 - odd one out - action - 2419 triplets - dimensional

In [12]:
df_acc_all = pd.DataFrame(data_all, columns = ['observer', 'task', 'image_set', 'd', 'accuracy', 'n'])
df_acc_all.to_csv(f"../data/subsampling_analysis-triplet_accuracy_alldata.csv", index=False)